# Materials Project API Examples
AI for Materials Science — Hands-on session 1, appendix

The in-class material runs through sections A–D of
`2026_2_Hands_on_session1_inclass.ipynb`.
This notebook is a **reference collection**, so there is no need to run it top to bottom.
Pick the section you need. The one exception is that **section 0 must always be run first.**

## What is in here

### 1 · Surveying the available routes
List the helper functions and search routes that `MPRester` exposes.

### 2 · Downloading a structure, inspecting it, and saving it as CIF
### 3 · Tracing provenance (thermo · provenance · task)
### 4 · Finding candidates that actually have the property data (elastic · dielectric · piezoelectric · magnetic)
### 5 · Electronic band structure and DOS
### 6 · Phonon band structure and DOS
### 7 · XAS and charge density
### 8 · Electrodes · surfaces · natural-language structure descriptions
### 9 · Stability in water: Pourbaix diagrams
### 10 · Extending to other chemical systems and to quaternary phase diagrams

---

### Before you run anything
API function names and arguments can change between versions, and **not every material has every
property.**
So each example starts by checking with `available_fields` and `inspect.signature`.

An empty result is still a result. If there is no data, record that too.

Details are in the
[MPRester documentation](https://materialsproject.github.io/api/_autosummary/mp_api.client.mprester.MPRester.html)
and the [official API examples](https://docs.materialsproject.org/downloading-data/using-the-api/examples).

## 0. Setup

### 0-1. Install the libraries
`plotly` is included because the quaternary phase diagram is saved as an interactive figure.

In [ ]:
!pip install -q pymatgen mp_api plotly

### 0-2. Import the libraries
This brings in everything the examples below share.
The electronic structure, phonon and Pourbaix tools are heavier, so those are imported inside the
sections that need them.

In [ ]:
import os
import inspect
from pathlib import Path
from getpass import getpass

import pandas as pd
import matplotlib.pyplot as plt

from pymatgen.core import Element
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDPlotter
from mp_api.client import MPRester

### 0-3. API key and output folder
Every example here queries the Materials Project directly, so a key is required.
You can get one from the [MP account page](https://next-gen.materialsproject.org/api).

In [ ]:
API_KEY = os.getenv("MP_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass("Materials Project API key: ").strip()

if not API_KEY:
    raise ValueError("An MP API key is required. Get one, then rerun this cell.")

OUTPUT = Path("outputs/02_api_examples")
OUTPUT.mkdir(parents=True, exist_ok=True)

with MPRester(API_KEY) as mpr:
    print("MP DB version:", mpr.db_version)

### 0-4. Decide whether to fetch the large files
Electronic structure, phonons, charge density, Pourbaix diagrams and the quaternary phase diagram are
large and slow.
The heavy cells in sections 5, 6, 7, 9 and 10 only run when the setting below is `True`.

Start with `False` and work through the lighter examples; switch it to `True` and rerun this cell
when you need them.

In [ ]:
RUN_LARGE_DOWNLOADS = False
print("Large downloads:", "enabled" if RUN_LARGE_DOWNLOADS else "skipped")

## 1. Surveying the available routes

The MP API has different doors depending on what you are after. Roughly:

- **Search by composition, structure, symmetry or property** — `materials.summary`,
  `get_structure_by_material_id`
- **Calculated energies, corrections, convex hull** — `materials.thermo`, `get_entries`,
  `get_entries_in_chemsys`
- **Individual calculations, provenance, papers** — `materials.tasks`, `materials.provenance`,
  `get_material_id_references`
- **Elastic, dielectric, piezoelectric, magnetic** — `materials.elasticity`, `dielectric`,
  `piezoelectric`, `magnetism`
- **Electronic band structure / DOS** — `get_bandstructure_by_material_id`,
  `get_dos_by_material_id`
- **Phonon band structure / DOS** — `get_phonon_bandstructure_by_material_id`,
  `get_phonon_dos_by_material_id`
- **XAS and charge density** — `materials.xas`, `get_charge_density_from_material_id`
- **Batteries** — `materials.insertion_electrodes`, `conversion_electrodes`
- **Surfaces and morphology** — `materials.surface_properties`, `get_wulff_shape`
- **Aqueous stability** — `get_pourbaix_entries` → `PourbaixDiagram`
- **Natural-language structure descriptions** — `materials.robocrys`

### 1-1. Checking what the installed version actually offers
**The installed version beats the documentation.** Get into the habit of listing the names yourself
with `dir`.

In [ ]:
## Names starting with get_ are the helper functions you can call directly.
print("MPRester get_* helpers:")
print([name for name in dir(MPRester) if name.startswith("get_")])

In [ ]:
with MPRester(API_KEY) as mpr:
    ## Skip the underscore-prefixed internal names and sort the rest.
    route_names = sorted(name for name in dir(mpr.materials) if not name.startswith("_"))
    print("Search routes under materials:")
    print(route_names)

In [ ]:
## Pick one route and check what conditions it takes and what it can return.
with MPRester(API_KEY) as mpr:
    print("elasticity.search conditions:", inspect.signature(mpr.materials.elasticity.search))
    print()
    print("elasticity output fields:", mpr.materials.elasticity.available_fields)

## 2. Downloading a structure, inspecting it, and saving it as CIF

`get_structure_by_material_id` turns a single material ID straight into a pymatgen `Structure`.
As practice we fetch silicon, `mp-149`, and check its atom count, lattice, density and space group.

`symprec` is the tolerance in Å allowed on atomic positions when determining symmetry.
**Even for a structure that already comes standardised from the API, this value can change the space
group that is reported.**
So whenever you report a space group, report the `symprec` you used with it.

In [ ]:
with MPRester(API_KEY) as mpr:
    structure = mpr.get_structure_by_material_id("mp-149")

print(structure.composition.reduced_formula, "| number of sites:", len(structure))
print("Lattice lengths (A):", structure.lattice.abc)
print("Density (g/cm^3):", float(structure.density))

In [ ]:
## Try different symprec values and see whether the determination changes.
for symprec in [0.01, 0.1]:
    analyzer = SpacegroupAnalyzer(structure, symprec=symprec)
    print(f"symprec={symprec}: {analyzer.get_space_group_symbol()} "
          f"({analyzer.get_space_group_number()})")

In [ ]:
## Saved as CIF, the structure opens in programs such as VESTA.
structure.to(filename=str(OUTPUT / "mp-149_Si.cif"), fmt="cif")
print("Saved:", OUTPUT / "mp-149_Si.cif")

## 3. Tracing provenance: thermo · provenance · task

**A single material has several calculations behind it.**
The point of this section is to trace where a value shown in a summary actually came from.

Look at `thermo_type` too. `GGA_GGA+U_R2SCAN` is thermodynamic data assembled from a mix of
calculations; **it is not another name for pure r2SCAN.**

In [ ]:
with MPRester(API_KEY) as mpr:
    thermo_docs = mpr.materials.thermo.search(
        material_ids=["mp-149"],
        fields=["material_id", "thermo_type", "formation_energy_per_atom", "energy_above_hull"],
        all_fields=False, num_chunks=1, chunk_size=20)

## model_dump turns a document object into a dictionary. Energies are in eV/atom.
pd.DataFrame([doc.model_dump() for doc in thermo_docs])

In [ ]:
with MPRester(API_KEY) as mpr:
    ## calc_types tells you what kinds of calculation are attached to this material.
    core_docs = mpr.materials.search(material_ids=["mp-149"],
                                     fields=["material_id", "calc_types"], all_fields=False)
    ## provenance carries the original literature references.
    provenance_docs = mpr.materials.provenance.search(material_ids=["mp-149"],
                                                      fields=["material_id", "references"],
                                                      all_fields=False)

print("calc_types:", [doc.calc_types for doc in core_docs])
print("provenance documents:", len(provenance_docs))

### 3-1. Following `origins` in a summary down to the calculation
`origins` tells you which task a given property value came from.
From there you can pull the task ID and inspect the inputs and outputs of that individual
calculation.

In [ ]:
with MPRester(API_KEY) as mpr:
    origin_docs = mpr.materials.summary.search(material_ids=["mp-149"],
                                               fields=["material_id", "origins"],
                                               all_fields=False)
    ## There may be no document, so fall back to an empty list.
    origins = origin_docs[0].origins if origin_docs else []
    print("Property origins:", origins)

    ## Keep only origins that carry a task_id, and take the first two.
    task_ids = [str(o.task_id) for o in origins if getattr(o, "task_id", None)][:2]
    task_docs = (mpr.materials.tasks.search(task_ids=task_ids,
                                            fields=["task_id", "input", "output"],
                                            all_fields=False) if task_ids else [])

print("Tasks retrieved:", [str(doc.task_id) for doc in task_docs])

## 4. Finding candidates that actually have the property data

To search on a property you first have to find **materials for which that property exists**.
That is what `has_props` is for.

`has_props` (does the data exist) and a property range condition (what is the value) are different
things.
**A value of zero and a missing value also have to be told apart.**

Below we find three oxygen-containing candidates per property, then retrieve documents from each
detail endpoint.
Note that the number of candidate IDs and the number of detail documents are not always the same.

In [ ]:
## Pairs of property name and search route name. The same procedure repeats for four properties.
PROPERTY_ROUTES = [("elasticity", "elasticity"), ("dielectric", "dielectric"),
                   ("piezoelectric", "piezoelectric"), ("magnetism", "magnetism")]

property_examples = {}
with MPRester(API_KEY) as mpr:
    for prop, route_name in PROPERTY_ROUTES:
        candidates = mpr.materials.summary.search(
            elements=["O"], has_props=[prop],
            fields=["material_id", "formula_pretty"], all_fields=False,
            num_chunks=1, chunk_size=3)
        ids = [str(d.material_id) for d in candidates]

        ## getattr fetches the search route matching the route name string.
        route = getattr(mpr.materials, route_name)
        detail = route.search(material_ids=ids, num_chunks=1, chunk_size=3) if ids else []
        property_examples[prop] = detail
        print(f"{prop:14s} {len(ids)} candidates -> {len(detail)} detail documents  {ids}")

In [ ]:
## Knowing what each route returns makes it easier to narrow fields later.
with MPRester(API_KEY) as mpr:
    for prop, route_name in PROPERTY_ROUTES:
        fields = getattr(mpr.materials, route_name).available_fields
        print(f"{prop:14s} ({len(fields)} fields): {fields[:8]} ...")

## 5. Electronic band structure and DOS

These files are large and take time to download and plot. **Set `RUN_LARGE_DOWNLOADS` to `True` in
0-4 to run this.**

The two figures are different views of the same calculation.

- **band structure** — energies along a path connecting high-symmetry k-points
- **DOS** — the same information gathered from all k-points and binned along the energy axis

Do not read a calculated band gap as an experimental value.
GGA-family functionals tend to underestimate band gaps.

In [ ]:
if RUN_LARGE_DOWNLOADS:
    from pymatgen.electronic_structure.plotter import BSPlotter, DosPlotter

    with MPRester(API_KEY) as mpr:
        bs = mpr.get_bandstructure_by_material_id("mp-149")
        dos = mpr.get_dos_by_material_id("mp-149")

    ## None means there is no data. Only plot when there is.
    if bs is not None:
        BSPlotter(bs).get_plot()
        plt.show()
        print("band gap:", bs.get_band_gap())

    if dos is not None:
        dp = DosPlotter()
        dp.add_dos("Si total DOS", dos)
        dp.get_plot()
        plt.show()
else:
    print("Skipped because RUN_LARGE_DOWNLOADS is False.")

## 6. Phonon band structure and DOS

Phonon data does not exist for every material.
So the order here is **check whether a document exists first**, and only then download.

Watch for **negative (imaginary) frequencies** in the plot.
They signal either dynamical instability or a calculation that has not converged well enough.

In [ ]:
if RUN_LARGE_DOWNLOADS:
    from pymatgen.phonon.plotter import PhononBSPlotter, PhononDosPlotter

    with MPRester(API_KEY) as mpr:
        summary_ph = mpr.materials.summary.search(material_ids=["mp-149"],
                                                  fields=["material_id", "phonon_IDs"],
                                                  all_fields=False)
        ## Fall back to an empty list if phonon_IDs is absent or empty.
        dfpt_ids = ((summary_ph[0].phonon_IDs or {}).get("dfpt", []) if summary_ph else [])

        if dfpt_ids:
            phonon_docs = mpr.materials.phonon.search(identifiers=dfpt_ids[:1],
                                                      fields=["identifier"], all_fields=False,
                                                      num_chunks=1, chunk_size=1)
            print("DFPT phonon identifiers:", [doc.identifier for doc in phonon_docs])
            ph_bs = mpr.get_phonon_bandstructure_by_material_id("mp-149")
            ph_dos = mpr.get_phonon_dos_by_material_id("mp-149")
        else:
            ph_bs, ph_dos = None, None
            print("This material has no phonon document. Try a different ID.")

    if ph_bs is not None:
        PhononBSPlotter(ph_bs).get_plot()
        plt.show()
    if ph_dos is not None:
        pp = PhononDosPlotter()
        pp.add_dos("Si phonon DOS", ph_dos)
        pp.get_plot()
        plt.show()
else:
    print("Skipped because RUN_LARGE_DOWNLOADS is False.")

## 7. XAS and charge density

For XAS (X-ray absorption spectroscopy) you have to specify both the **absorbing element** and the
**absorption edge**.
A composition alone does not say which element or which edge you mean.

Below is the Ti K-edge spectrum of TiO₂. Check the units on the axes:
x is photon energy in eV, y is relative intensity in arbitrary units.

In [ ]:
with MPRester(API_KEY) as mpr:
    ## Element("Ti") builds an element object to name the absorbing species.
    spectra = mpr.materials.xas.search(formula="TiO2", absorbing_element=Element("Ti"),
                                       edge="K", num_chunks=1, chunk_size=1)

print("Ti K-edge documents:", len(spectra))

if spectra:
    spectrum = spectra[0].spectrum
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(spectrum.x, spectrum.y)
    ax.set(xlabel="Photon energy (eV)", ylabel="Intensity (a.u.)", title="Ti K-edge XAS")
    fig.tight_layout()
    plt.show()

Charge density is a value on a three-dimensional grid, so the files are very large. It is only
fetched when the large-download setting is on.
CHGCAR, the format it is written in, is the charge density file format used by VASP.

In [ ]:
if RUN_LARGE_DOWNLOADS:
    with MPRester(API_KEY) as mpr:
        charge = mpr.get_charge_density_from_material_id("mp-149")

    if charge is not None:
        charge.write_file(str(OUTPUT / "mp-149_CHGCAR"))
        print("Saved:", OUTPUT / "mp-149_CHGCAR")
    else:
        print("This material has no charge density data.")
else:
    print("Skipped because RUN_LARGE_DOWNLOADS is False.")

## 8. Electrodes · surfaces · natural-language structure descriptions

**Different endpoints can use different ID schemes.**
Electrode results in particular carry a battery ID and charged/discharged state information rather
than a material ID, so check the column names in the output first.

In [ ]:
with MPRester(API_KEY) as mpr:
    ## Electrode candidates with Li as the working ion, containing Fe, P and O, at 2.5-4.5 V average.
    batteries = mpr.materials.insertion_electrodes.search(
        elements=["Fe", "P", "O"], working_ion=Element("Li"),
        average_voltage=(2.5, 4.5), num_chunks=1, chunk_size=3)

print("Li insertion electrode documents:", len(batteries))
pd.DataFrame([doc.model_dump() for doc in batteries]).head() if batteries else "No results"

In [ ]:
with MPRester(API_KEY) as mpr:
    ## Surface properties are looked up by material ID.
    surfaces = mpr.materials.surface_properties.search(material_ids=["mp-149"],
                                                       num_chunks=1, chunk_size=1)
    ## robocrys describes a structure in sentences a person can read.
    descriptions = mpr.materials.robocrys.search(keywords=["perovskite"],
                                                 num_chunks=1, chunk_size=1)

print("Si surface documents:", len(surfaces))
print("Structure description results:", len(descriptions))

## 9. Stability in water: Pourbaix diagrams

The phase diagrams so far dealt only with solids competing against each other.
A Pourbaix diagram adds **pH and potential** as axes, showing what is stable in an aqueous
environment.

Because dissolved ions are involved, **you must state the ion concentration.**
Change the concentration and the boundaries of the stability regions move.

The example below sets the Fe concentration to 10⁻⁶ mol/L, a common reference concentration in
corrosion studies.

In [ ]:
if RUN_LARGE_DOWNLOADS:
    from pymatgen.analysis.pourbaix_diagram import PourbaixDiagram, PourbaixPlotter

    with MPRester(API_KEY) as mpr:
        aqueous_entries = mpr.get_pourbaix_entries(["Fe"])
    print("Aqueous entries:", len(aqueous_entries))

    ## conc_dict is in mol/L.
    aqueous_pd = PourbaixDiagram(aqueous_entries, conc_dict={"Fe": 1e-6})

    ## In limits, the first pair is the pH range and the second is the potential range in V.
    PourbaixPlotter(aqueous_pd).get_pourbaix_plot(limits=[[0, 14], [-2, 2]])
    plt.show()
else:
    print("Skipped because RUN_LARGE_DOWNLOADS is False.")

## 10. Extending to other chemical systems and to quaternary phase diagrams

In section C of the class we drew Li–Fe–O from a bundled dataset.
Querying MP directly lets you draw **any chemical system you like**.

### 10-1. The Li–Co–O ternary
`get_entries_in_chemsys` brings back **every subsystem** of the elements you name.
As explained in section C, leaving those out makes the convex hull impossible to build.

In [ ]:
PD_ELEMENTS = ["Li", "Co", "O"]
THERMO_TYPE = "GGA_GGA+U"

with MPRester(API_KEY) as mpr:
    ## thermo_types states which calculation set to use. Mixing them changes the energy reference.
    entries = mpr.get_entries_in_chemsys(PD_ELEMENTS, compatible_only=True,
                                         additional_criteria={"thermo_types": [THERMO_TYPE]})
    db_version = mpr.db_version

print(f"{'-'.join(PD_ELEMENTS)} | {THERMO_TYPE} | DB {db_version}")
print("Entries:", len(entries))

In [ ]:
phase_diagram = PhaseDiagram(entries)
print("Stable entries:", len(phase_diagram.stable_entries))

ax = PDPlotter(phase_diagram, backend="matplotlib",
               show_unstable=False).get_plot(label_stable=True)
ax.figure.set_size_inches(8, 7)
ax.figure.savefig(OUTPUT / f"{'-'.join(PD_ELEMENTS)}_phase_diagram.png",
                  dpi=180, bbox_inches="tight")
plt.show()

### 10-2. The Li–Fe–P–O quaternary
With four elements a single triangle is no longer enough; the shape becomes a tetrahedron, which
needs a three-dimensional figure.
Building it with `backend="plotly"` produces an HTML file you can rotate in a browser.

**This query returns a lot of entries.** It takes a while, so it only runs with the large-download
setting on.

Do not cut the volume down by filtering on `is_stable=True` or by using only part of the paged
results.
A hull missing its competing phases reports the wrong stability.

In [ ]:
if RUN_LARGE_DOWNLOADS:
    with MPRester(API_KEY) as mpr:
        lfp_entries = mpr.get_entries_in_chemsys(
            ["Li", "Fe", "P", "O"],
            additional_criteria={"thermo_types": ["GGA_GGA+U"]})
    print("Entries:", len(lfp_entries))

    lfp_pd = PhaseDiagram(lfp_entries)
    lfp_plot = PDPlotter(lfp_pd, backend="plotly", show_unstable=False).get_plot()

    ## Saved as HTML so you can reopen it in a browser later.
    html_path = OUTPUT / "Li-Fe-P-O_4component_phase_diagram.html"
    lfp_plot.write_html(str(html_path))
    print("Saved:", html_path)
    lfp_plot.show()
else:
    print("Skipped because RUN_LARGE_DOWNLOADS is False.")